# Colab One-Click + Phase Cells

This notebook supports two styles:
1. **One-click run** (recommended): run Cell 5 only after setup.
2. **Phase-by-phase**: run Cells 6 → 7 → 8, then choose 9 (GAN) or 10 (CFM), and optionally 11 (overfit check).

## CFM condition (default for Stage 3)
`c = [hidden_state, direction, DI]`
- **hidden_state**: from author GRU checkpoint (`extractor_path`)
- **direction / DI**: from original `label.csv` (author dataset labels)

Upload author checkpoints to skip Stage 1/2:
- `results/HystereticPrediction/HystereticGRU/2024-06-12_18-13-47`
- `results/DamageQuantification/vit_b_16/2024-06-12_14-01-03`

## Which cell to run?
- One-click CFM overfit (recommended): **Cell 5** (`MODE="overfit"`, `BACKEND="cfm"`)
- Manual CFM overfit with author ckpt: **Cells 1–4, 8, 11** (skip 6/7 if author results exist)

Other phase cells are preserved below for manual control.


In [ ]:
# === EDIT THIS IF NEEDED ===
PROJECT_DIR = "/content/intership-project"

import os
assert os.path.isdir(PROJECT_DIR), f"Project dir not found: {PROJECT_DIR}"
%cd /content/intership-project
print("Project dir:", PROJECT_DIR)


In [ ]:
# Install dependencies
!pip install -q --upgrade pip
!pip install -q -r requirements.txt


In [ ]:
# Optional: Weights & Biases
USE_WANDB = True  # set False to disable

if USE_WANDB:
    import wandb
    wandb.login()
    WANDB_FLAG = "--use_wandb"
else:
    WANDB_FLAG = ""

print("WANDB_FLAG:", WANDB_FLAG)


In [ ]:
# Data quick check (must exist before training)
required_paths = [
    "Data/Hysteresis/info_w_mander_big.csv",
    "Data/Hysteresis/id_list_small.json",
    "Data/DamagePattern/label.csv",
    "Data/DamagePattern/id_list.json",
]

for p in required_paths:
    print(p, "OK" if os.path.exists(p) else "MISSING")


In [ ]:
# One-click pipeline runner (GAN / CFM / Overfit)
MODE = "overfit"      # "smoke" or "overfit"
BACKEND = "cfm"     # "gan" or "cfm"

wandb_opt = "--with-wandb" if USE_WANDB else ""
!bash scripts/colab_quickstart.sh {wandb_opt} --mode={MODE} --backend={BACKEND}


In [ ]:
# Stage 1 (manual): Hysteretic prediction
!python train_hysteresis.py --architecture HystereticGRU --epoch 5 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# Stage 2 (manual): Damage quantification
!python train_quantification.py --backbone vit_b_16 --epoch 5 --batch_size 4 --num_workers 1 {WANDB_FLAG}


In [ ]:
# Resolve Stage 1/2 paths: prefer author checkpoints, else latest local run
import glob

AUTHOR_EXTRACTOR = "results/HystereticPrediction/HystereticGRU/2024-06-12_18-13-47"
AUTHOR_REGRESSOR = "results/DamageQuantification/vit_b_16/2024-06-12_14-01-03"

def latest_run(base_glob):
    runs = sorted(glob.glob(base_glob), key=os.path.getmtime)
    assert runs, f"No runs found for: {base_glob}"
    return runs[-1]

def resolve_run_path(author_path, latest_glob):
    if os.path.isdir(author_path):
        return author_path
    return latest_run(latest_glob)

extractor_path = resolve_run_path(AUTHOR_EXTRACTOR, "results/HystereticPrediction/HystereticGRU/*")
regressor_path = resolve_run_path(AUTHOR_REGRESSOR, "results/DamageQuantification/vit_b_16/*")
print("extractor_path =", extractor_path)
print("regressor_path =", regressor_path)


In [ ]:
# Stage 3 (manual): GAN backend smoke
!python train_mixed.py \
  --architecture HystereticCGAN \
  --loss_type SGAN \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning False \
  --direction_conditioning False \
  --add_DI_loss False \
  --image_size 128 \
  --z_dim 100 \
  --G_h_size 128 \
  --D_h_size 128 \
  --SELU False \
  --spectral_D False \
  --no_batch_norm False \
  --lr_schedule warmup_cosine \
  --warmup_epochs 2 \
  --min_lr_ratio 0.1 \
  --epoch 5 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}


In [ ]:
# Stage 3 (manual): CFM ADM backend with Heun solver
!python train_mixed.py \
  --architecture CFMADM \
  --cfm_solver heun \
  --cfm_ode_steps 20 \
  --cfm_model_channels 128 \
  --cfm_num_res_blocks 2 \
  --cfm_cond_dim 512 \
  --cfm_dropout 0.0 \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning True \
  --direction_conditioning True \
  --include_di_conditioning True \
  --add_DI_loss False \
  --image_size 128 \
  --lr_schedule warmup_cosine \
  --warmup_epochs 2 \
  --min_lr_ratio 0.1 \
  --epoch 5 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}


In [ ]:
# Extra verification: overfit a single sample for 100 epochs (CFM + warmup cosine)
!python train_mixed.py \
  --architecture CFMADM \
  --cfm_solver heun \
  --cfm_ode_steps 20 \
  --extractor_path "{extractor_path}" \
  --regressor_path "{regressor_path}" \
  --data_path id_list.json \
  --latent_conditioning True \
  --direction_conditioning True \
  --include_di_conditioning True \
  --add_DI_loss False \
  --image_size 128 \
  --lr_schedule warmup_cosine \
  --warmup_epochs 10 \
  --min_lr_ratio 0.1 \
  --overfit_single_sample \
  --overfit_sample_index 0 \
  --overfit_repeat_steps 32 \
  --epoch 100 \
  --batch_size 4 \
  --num_workers 1 \
  {WANDB_FLAG}
